# Preprocess GEO Datasets for Mouse Skin Atlas

This notebook loads, preprocesses, and exports GEO scRNA-seq datasets into the same
standardized format used by the internal datasets (counts.mtx, gene_names.csv, metadata.csv).

**Output format** (matching internal datasets):
- `counts_<name>.mtx` - Matrix Market sparse matrix (genes x cells)
- `gene_names_<name>.csv` - One gene name per line, no header
- `metadata_<name>.csv` - R-style CSV with cell barcodes as index

## Datasets:
1. **GSE142471 (Haensel)** - Epidermal basal cells, wound healing Day 4 (5 samples)
2. **GSE113854 (Guerrero-Juarez)** - Fibroblast heterogeneity, wound Day 12 (1 merged sample)
3. **GSE186527 (Mascharak)** - Scarring vs regenerative healing (9 timepoint samples)
4. **GSE218430 (Liu)** - LncRNA SNHG26 wound healing WT vs KO (4 samples)
5. **GSE178758 (Foster)** - Dermal fibroblast scRNA-seq Inner/Outer wound (11 samples)

In [1]:
import gzip
import re
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.sparse import csr_matrix
import anndata as ad
import scanpy as sc
import warnings

warnings.filterwarnings('ignore')
sc.settings.verbosity = 3

DATA_DIR = Path("data")
print(f"Data directory: {DATA_DIR.resolve()}")

Data directory: /home/scumpia-mrl/Desktop/Sujit/Projects/mice-skin-atlas/integration_scvi/data


## Helper Functions

In [2]:
def load_10x_mtx_gzip(
    barcodes_file: Path,
    features_file: Path,
    matrix_file: Path,
    sample_name: str = "",
) -> ad.AnnData:
    """
    Load a 10x sample from gzipped barcodes/features/matrix files.
    Returns AnnData with cells x genes.
    """
    # Read barcodes
    with gzip.open(barcodes_file, 'rt') as f:
        barcodes = [line.strip().split('\t')[0] for line in f]

    # Read features/genes
    with gzip.open(features_file, 'rt') as f:
        lines = [line.strip().split('\t') for line in f]
        gene_ids = [l[0] for l in lines]
        gene_names = [l[1] for l in lines] if len(lines[0]) >= 2 else gene_ids

    # Read matrix
    with gzip.open(matrix_file, 'rb') as f:
        matrix = sio.mmread(f)

    # Transpose from genes x cells to cells x genes
    if matrix.shape[0] == len(gene_ids) and matrix.shape[1] == len(barcodes):
        matrix = matrix.T
    matrix = csr_matrix(matrix)

    # Add sample prefix to barcodes
    if sample_name:
        barcodes = [f"{sample_name}_{bc}" for bc in barcodes]

    adata = ad.AnnData(
        X=matrix,
        obs=pd.DataFrame(index=barcodes),
        var=pd.DataFrame({'gene_ids': gene_ids}, index=gene_names),
    )
    adata.var_names_make_unique()

    return adata


def basic_qc_filter(
    adata: ad.AnnData,
    min_genes: int = 100,
    min_cells: int = 3,
    max_pct_mito: float = 25.0,
) -> ad.AnnData:
    """
    Basic QC filtering: remove low-quality cells and lowly-expressed genes.
    """
    adata = adata.copy()
    n_cells_before = adata.n_obs
    n_genes_before = adata.n_vars

    # QC metrics
    adata.var['mt'] = adata.var_names.str.startswith(('mt-', 'MT-'))
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    # Filter
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    adata = adata[adata.obs['pct_counts_mt'] < max_pct_mito, :].copy()

    print(f"  QC: {n_cells_before:,} -> {adata.n_obs:,} cells, "
          f"{n_genes_before:,} -> {adata.n_vars:,} genes")
    return adata


def export_to_internal_format(
    adata: ad.AnnData,
    name: str,
    output_dir: Path,
):
    """
    Export AnnData to the internal dataset format:
      - counts_<name>.mtx  (genes x cells, Matrix Market)
      - gene_names_<name>.csv  (one gene per line, no header)
      - metadata_<name>.csv  (R-style CSV with quoted index)
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. counts: transpose to genes x cells for MTX
    counts_path = output_dir / f"counts_{name}.mtx"
    X = adata.X
    if not isinstance(X, csr_matrix):
        X = csr_matrix(X)
    sio.mmwrite(str(counts_path), X.T)  # genes x cells
    print(f"  Wrote {counts_path.name} ({adata.n_vars} genes x {adata.n_obs} cells)")

    # 2. gene names: one per line, no header
    genes_path = output_dir / f"gene_names_{name}.csv"
    with open(genes_path, 'w') as f:
        for gene in adata.var_names:
            f.write(f"{gene}\n")
    print(f"  Wrote {genes_path.name} ({len(adata.var_names)} genes)")

    # 3. metadata: R-style CSV with quoted index
    meta_path = output_dir / f"metadata_{name}.csv"
    meta = adata.obs.copy()
    # Remove scanpy QC columns that aren't needed downstream
    drop_cols = [c for c in meta.columns if c.startswith(('n_genes', 'n_cells', 'total_counts', 'pct_counts', 'mt'))]
    meta = meta.drop(columns=[c for c in drop_cols if c in meta.columns], errors='ignore')
    meta.to_csv(meta_path)
    print(f"  Wrote {meta_path.name} ({len(meta)} cells, columns: {list(meta.columns)})")

    return counts_path, genes_path, meta_path

---
## 1. GSE142471 - Haensel (Wound Day 4)

Epidermal basal cells in skin homeostasis and wound healing.  
5 samples: Un-Wounded_1, Un-Wounded_2, Wounded_1, Wounded_2, Wounded_3  
Naming: `GSM*_barcodes_<SampleName>_scRNA-Seq.tsv.gz`

In [3]:
haensel_dir = DATA_DIR / "Daniel-Haensel-GSE142471"
print(f"Haensel directory: {haensel_dir}")
print(f"Files: {sorted([f.name for f in haensel_dir.iterdir()])}")

Haensel directory: data/Daniel-Haensel-GSE142471
Files: ['GSM4230076_Un-Wounded_1_scRNA-Seq.mtx.gz', 'GSM4230076_barcodes_Un-Wounded_1_scRNA-Seq.tsv.gz', 'GSM4230076_genes_Un-Wounded_1_scRNA-Seq.tsv.gz', 'GSM4230077_Un-Wounded_2_scRNA-Seq.mtx.gz', 'GSM4230077_barcodes_Un-Wounded_2_scRNA-Seq.tsv.gz', 'GSM4230077_genes_Un-Wounded_2_scRNA-Seq.tsv.gz', 'GSM4230078_Wounded_1_scRNA-Seq.mtx.gz', 'GSM4230078_barcodes_Wounded_1_scRNA-Seq.tsv.gz', 'GSM4230078_genes_Wounded_1_scRNA-Seq.tsv.gz', 'GSM4230079_Wounded_2_scRNA-Seq.mtx.gz', 'GSM4230079_barcodes_Wounded_2_scRNA-Seq.tsv.gz', 'GSM4230079_genes_Wounded_2_scRNA-Seq.tsv.gz', 'GSM4230080_Wounded_3_scRNA-Seq.mtx.gz', 'GSM4230080_barcodes_Wounded_3_scRNA-Seq.tsv.gz', 'GSM4230080_genes_Wounded_3_scRNA-Seq.tsv.gz']


In [4]:
# Discover and load Haensel samples
# Naming pattern: GSM*_barcodes_<sample>_scRNA-Seq.tsv.gz
#                 GSM*_genes_<sample>_scRNA-Seq.tsv.gz
#                 GSM*_<sample>_scRNA-Seq.mtx.gz

haensel_samples = {}
gsm_files = {}
for f in haensel_dir.iterdir():
    if not f.is_file():
        continue
    match = re.match(r'(GSM\d+)_(.+)', f.name)
    if match:
        gsm_id = match.group(1)
        rest = match.group(2)
        if gsm_id not in gsm_files:
            gsm_files[gsm_id] = []
        gsm_files[gsm_id].append((rest, f))

for gsm_id, files in gsm_files.items():
    sample_files = {}
    sample_name = None
    for rest, filepath in files:
        if rest.startswith('barcodes_'):
            sample_files['barcodes'] = filepath
            name_match = re.match(r'barcodes_(.+?)_scRNA-Seq\.tsv', rest)
            if name_match:
                sample_name = name_match.group(1)
        elif rest.startswith('genes_'):
            sample_files['features'] = filepath
        elif rest.endswith('.mtx.gz'):
            sample_files['matrix'] = filepath
    if sample_name and len(sample_files) == 3:
        haensel_samples[sample_name] = sample_files

print(f"Discovered {len(haensel_samples)} samples: {list(haensel_samples.keys())}")

Discovered 5 samples: ['Un-Wounded_2', 'Wounded_3', 'Wounded_1', 'Wounded_2', 'Un-Wounded_1']


In [5]:
# Load all Haensel samples
haensel_adatas = []
for sample_name, files in haensel_samples.items():
    print(f"Loading {sample_name}...")
    adata = load_10x_mtx_gzip(
        files['barcodes'], files['features'], files['matrix'],
        sample_name=sample_name,
    )
    adata = basic_qc_filter(adata)

    # Add metadata
    adata.obs['Sample'] = sample_name
    if 'Un-Wounded' in sample_name or 'Unwounded' in sample_name:
        adata.obs['Type'] = 'Unwounded'
    else:
        adata.obs['Type'] = 'Wounded'
    adata.obs['Timepoint'] = 'D4'

    haensel_adatas.append(adata)
    print(f"  -> {adata.n_obs:,} cells, {adata.n_vars:,} genes")

# Concatenate
adata_haensel = ad.concat(haensel_adatas, join='outer')
adata_haensel.obs_names_make_unique()
adata_haensel.var_names_make_unique()
print(f"\nHaensel combined: {adata_haensel.n_obs:,} cells, {adata_haensel.n_vars:,} genes")
print(adata_haensel.obs['Type'].value_counts())

Loading Un-Wounded_2...
filtered out 11412 genes that are detected in less than 3 cells
  QC: 5,517 -> 5,506 cells, 27,998 -> 16,586 genes
  -> 5,506 cells, 16,586 genes
Loading Wounded_3...
filtered out 1 cells that have less than 100 genes expressed
filtered out 10969 genes that are detected in less than 3 cells
  QC: 4,152 -> 4,149 cells, 27,998 -> 17,029 genes
  -> 4,149 cells, 17,029 genes
Loading Wounded_1...
filtered out 10421 genes that are detected in less than 3 cells
  QC: 7,578 -> 7,576 cells, 27,998 -> 17,577 genes
  -> 7,576 cells, 17,577 genes
Loading Wounded_2...
filtered out 34 cells that have less than 100 genes expressed
filtered out 11188 genes that are detected in less than 3 cells
  QC: 4,698 -> 4,655 cells, 27,998 -> 16,810 genes
  -> 4,655 cells, 16,810 genes
Loading Un-Wounded_1...
filtered out 12221 genes that are detected in less than 3 cells
  QC: 5,372 -> 5,366 cells, 27,998 -> 15,777 genes
  -> 5,366 cells, 15,777 genes

Haensel combined: 27,252 cells, 18,

In [6]:
# Export Haensel
export_to_internal_format(adata_haensel, "haensel_wound_d4", DATA_DIR)

  Wrote counts_haensel_wound_d4.mtx (18478 genes x 27252 cells)
  Wrote gene_names_haensel_wound_d4.csv (18478 genes)
  Wrote metadata_haensel_wound_d4.csv (27252 cells, columns: ['Sample', 'Type', 'Timepoint'])


(PosixPath('data/counts_haensel_wound_d4.mtx'),
 PosixPath('data/gene_names_haensel_wound_d4.csv'),
 PosixPath('data/metadata_haensel_wound_d4.csv'))

---
## 2. GSE113854 - Guerrero-Juarez (Wound Day 12)

Fibroblast heterogeneity and myeloid-derived adipocyte progenitors.  
1 merged sample: GSM3121363  
Naming: `GSM3121363_{barcodes|genes|matrix}.tsv.gz`

In [7]:
guerrero_dir = DATA_DIR / "Guerrero-Juarez-GSE113854"
print(f"Guerrero-Juarez directory: {guerrero_dir}")
print(f"Files: {sorted([f.name for f in guerrero_dir.iterdir()])}")

Guerrero-Juarez directory: data/Guerrero-Juarez-GSE113854
Files: ['GSM3121363_barcodes.tsv.gz', 'GSM3121363_genes.tsv.gz', 'GSM3121363_matrix.mtx.gz']


In [8]:
# Load Guerrero-Juarez (single merged sample)
adata_guerrero = load_10x_mtx_gzip(
    guerrero_dir / "GSM3121363_barcodes.tsv.gz",
    guerrero_dir / "GSM3121363_genes.tsv.gz",
    guerrero_dir / "GSM3121363_matrix.mtx.gz",
    sample_name="GSM3121363",
)
print(f"Raw: {adata_guerrero.n_obs:,} cells, {adata_guerrero.n_vars:,} genes")

adata_guerrero = basic_qc_filter(adata_guerrero)

# Add metadata
adata_guerrero.obs['Sample'] = 'GSM3121363'
adata_guerrero.obs['Type'] = 'Wounded'
adata_guerrero.obs['Timepoint'] = 'D12'

print(f"\nGuerrero-Juarez: {adata_guerrero.n_obs:,} cells, {adata_guerrero.n_vars:,} genes")

Raw: 22,322 cells, 27,998 genes
filtered out 10908 genes that are detected in less than 3 cells
  QC: 22,322 -> 22,320 cells, 27,998 -> 17,090 genes

Guerrero-Juarez: 22,320 cells, 17,090 genes


In [9]:
# Export Guerrero-Juarez
export_to_internal_format(adata_guerrero, "guerrero_wound_d12", DATA_DIR)

  Wrote counts_guerrero_wound_d12.mtx (17090 genes x 22320 cells)
  Wrote gene_names_guerrero_wound_d12.csv (17090 genes)
  Wrote metadata_guerrero_wound_d12.csv (22320 cells, columns: ['Sample', 'Type', 'Timepoint'])


(PosixPath('data/counts_guerrero_wound_d12.mtx'),
 PosixPath('data/gene_names_guerrero_wound_d12.csv'),
 PosixPath('data/metadata_guerrero_wound_d12.csv'))

---
## 3. GSE186527 - Mascharak (Scarring vs Regeneration)

Scarring vs regenerative wound healing with YAP inhibition.  
9 samples at various timepoints with Pulastilla (P) and Vehicle (V) treatments.  
Naming: `GSM*_<Timepoint>_{barcodes|features|matrix}.tsv.gz`

In [10]:
mascharak_dir = DATA_DIR / "Shamik-Mascharak-GSE186527"
print(f"Mascharak directory: {mascharak_dir}")
print(f"Files: {sorted([f.name for f in mascharak_dir.iterdir()])}")

Mascharak directory: data/Shamik-Mascharak-GSE186527
Files: ['GSM5429653_T1_barcodes.tsv.gz', 'GSM5429653_T1_features.tsv.gz', 'GSM5429653_T1_matrix.mtx.gz', 'GSM5429654_T2_barcodes.tsv.gz', 'GSM5429654_T2_features.tsv.gz', 'GSM5429654_T2_matrix.mtx.gz', 'GSM5429655_T3_barcodes.tsv.gz', 'GSM5429655_T3_features.tsv.gz', 'GSM5429655_T3_matrix.mtx.gz', 'GSM5429656_T-7P_barcodes.tsv.gz', 'GSM5429656_T-7P_features.tsv.gz', 'GSM5429656_T-7P_matrix.mtx.gz', 'GSM5429657_T-7V_barcodes.tsv.gz', 'GSM5429657_T-7V_features.tsv.gz', 'GSM5429657_T-7V_matrix.mtx.gz', 'GSM5429658_T-14P_barcodes.tsv.gz', 'GSM5429658_T-14P_features.tsv.gz', 'GSM5429658_T-14P_matrix.mtx.gz', 'GSM5429659_T-14V_barcodes.tsv.gz', 'GSM5429659_T-14V_features.tsv.gz', 'GSM5429659_T-14V_matrix.mtx.gz', 'GSM5429660_T-30P_barcodes.tsv.gz', 'GSM5429660_T-30P_features.tsv.gz', 'GSM5429660_T-30P_matrix.mtx.gz', 'GSM5429661_T-30V_barcodes.tsv.gz', 'GSM5429661_T-30V_features.tsv.gz', 'GSM5429661_T-30V_matrix.mtx.gz']


In [18]:
import re

# Discover Mascharak samples
# Examples:
# GSM5429653_T1_barcodes.tsv.gz
# GSM5429653_T1_features.tsv.gz
# GSM5429653_T1_matrix.mtx.gz

mascharak_samples = {}

pattern = re.compile(
    r'GSM\d+_(.+?)_(barcodes|features)\.tsv\.gz|GSM\d+_(.+?)_(matrix)\.mtx\.gz'
)

for f in mascharak_dir.iterdir():
    m = pattern.match(f.name)
    if not m:
        continue

    # Handle the two alternative capture groups
    if m.group(1) is not None:
        sample_name = m.group(1)
        file_type = m.group(2)
    else:
        sample_name = m.group(3)
        file_type = m.group(4)

    if sample_name not in mascharak_samples:
        mascharak_samples[sample_name] = {}

    mascharak_samples[sample_name][file_type] = f

# Keep only complete samples
mascharak_samples = {
    k: v for k, v in mascharak_samples.items()
    if all(t in v for t in ['barcodes', 'features', 'matrix'])
}

print(f"Discovered {len(mascharak_samples)} samples: {sorted(mascharak_samples.keys())}")


Discovered 9 samples: ['T-14P', 'T-14V', 'T-30P', 'T-30V', 'T-7P', 'T-7V', 'T1', 'T2', 'T3']


In [19]:
# Parse condition from Mascharak sample names
def parse_mascharak_condition(sample_name: str) -> dict:
    """Parse timepoint and treatment from sample names like T1, T-7P, T-14V."""
    # Simple timepoints (T1, T2, T3) = baseline
    m = re.match(r'^T(\d+)$', sample_name)
    if m:
        return {'Timepoint': f'D{m.group(1)}', 'Treatment': 'Baseline', 'Type': f'Baseline_D{m.group(1)}'}
    
    # Treated timepoints (T-7P, T-14V, T-30P, etc.)
    m = re.match(r'^T-(\d+)([PV])$', sample_name)
    if m:
        day = m.group(1)
        treat = 'Verteporfin' if m.group(2) == 'V' else 'Pulastilla'
        return {'Timepoint': f'D{day}', 'Treatment': treat, 'Type': f'{treat}_D{day}'}
    
    return {'Timepoint': sample_name, 'Treatment': 'Unknown', 'Type': sample_name}


# Load all Mascharak samples
mascharak_adatas = []
for sample_name in sorted(mascharak_samples.keys()):
    files = mascharak_samples[sample_name]
    print(f"Loading {sample_name}...")
    adata = load_10x_mtx_gzip(
        files['barcodes'], files['features'], files['matrix'],
        sample_name=sample_name,
    )
    adata = basic_qc_filter(adata)

    # Add metadata
    adata.obs['Sample'] = sample_name
    meta = parse_mascharak_condition(sample_name)
    for k, v in meta.items():
        adata.obs[k] = v

    mascharak_adatas.append(adata)
    print(f"  -> {adata.n_obs:,} cells, {adata.n_vars:,} genes")

# Concatenate
adata_mascharak = ad.concat(mascharak_adatas, join='outer')
adata_mascharak.obs_names_make_unique()
adata_mascharak.var_names_make_unique()
print(f"\nMascharak combined: {adata_mascharak.n_obs:,} cells, {adata_mascharak.n_vars:,} genes")
print(adata_mascharak.obs['Type'].value_counts())

Loading T-14P...
filtered out 64 cells that have less than 100 genes expressed
filtered out 15785 genes that are detected in less than 3 cells
  QC: 906 -> 806 cells, 31,063 -> 15,278 genes
  -> 806 cells, 15,278 genes
Loading T-14V...
filtered out 51 cells that have less than 100 genes expressed
filtered out 16534 genes that are detected in less than 3 cells
  QC: 470 -> 391 cells, 31,063 -> 14,529 genes
  -> 391 cells, 14,529 genes
Loading T-30P...
filtered out 56 cells that have less than 100 genes expressed
filtered out 18021 genes that are detected in less than 3 cells
  QC: 474 -> 413 cells, 31,063 -> 13,042 genes
  -> 413 cells, 13,042 genes
Loading T-30V...
filtered out 36 cells that have less than 100 genes expressed
filtered out 18284 genes that are detected in less than 3 cells
  QC: 452 -> 409 cells, 31,063 -> 12,779 genes
  -> 409 cells, 12,779 genes
Loading T-7P...
filtered out 27 cells that have less than 100 genes expressed
filtered out 17105 genes that are detected in 

In [20]:
# Export Mascharak
export_to_internal_format(adata_mascharak, "mascharak_regeneration", DATA_DIR)

  Wrote counts_mascharak_regeneration.mtx (18000 genes x 8793 cells)
  Wrote gene_names_mascharak_regeneration.csv (18000 genes)
  Wrote metadata_mascharak_regeneration.csv (8793 cells, columns: ['Sample', 'Timepoint', 'Treatment', 'Type'])


(PosixPath('data/counts_mascharak_regeneration.mtx'),
 PosixPath('data/gene_names_mascharak_regeneration.csv'),
 PosixPath('data/metadata_mascharak_regeneration.csv'))

---
## 4. GSE218430 - Liu (SNHG26 WT vs KO)

LncRNA SNHG26 in wound healing.  
4 samples: KoSkin1, KoWound1, WtSkin1, WtWound1  
Data is in nested extracted tar.gz directories.

In [21]:
liu_dir = DATA_DIR / "Zhuang-Liu-GSE218430"
print(f"Liu directory: {liu_dir}")

# Find extracted 10x directories
liu_samples = {}
for subdir in liu_dir.iterdir():
    if subdir.is_dir() and 'filtered_feature_bc_matrix' in subdir.name:
        # Find the actual barcodes/features/matrix inside nested dirs
        for mtx_file in subdir.rglob('matrix.mtx.gz'):
            parent = mtx_file.parent
            barcodes = parent / 'barcodes.tsv.gz'
            features = parent / 'features.tsv.gz'
            if barcodes.exists() and features.exists():
                # Extract sample name from directory
                name_match = re.match(r'GSE218430_(.+?)_filtered', subdir.name)
                if name_match:
                    sample_name = name_match.group(1)
                    liu_samples[sample_name] = {
                        'barcodes': barcodes,
                        'features': features,
                        'matrix': mtx_file,
                    }

print(f"Discovered {len(liu_samples)} samples: {sorted(liu_samples.keys())}")

Liu directory: data/Zhuang-Liu-GSE218430
Discovered 4 samples: ['KoSkin1', 'KoWound1', 'WtSkin1', 'WtWound1']


In [22]:
# Parse condition from Liu sample names
def parse_liu_condition(sample_name: str) -> dict:
    """Parse genotype and tissue from names like KoSkin1, WtWound1."""
    m = re.match(r'(Ko|Wt)(Skin|Wound)(\d*)', sample_name, re.IGNORECASE)
    if m:
        genotype = 'KO' if m.group(1).lower() == 'ko' else 'WT'
        tissue = m.group(2)
        return {'Genotype': genotype, 'Tissue': tissue, 'Type': f'{genotype}_{tissue}'}
    return {'Genotype': 'Unknown', 'Tissue': 'Unknown', 'Type': sample_name}


# Load all Liu samples
liu_adatas = []
for sample_name in sorted(liu_samples.keys()):
    files = liu_samples[sample_name]
    print(f"Loading {sample_name}...")
    adata = load_10x_mtx_gzip(
        files['barcodes'], files['features'], files['matrix'],
        sample_name=sample_name,
    )
    adata = basic_qc_filter(adata)

    # Add metadata
    adata.obs['Sample'] = sample_name
    meta = parse_liu_condition(sample_name)
    for k, v in meta.items():
        adata.obs[k] = v

    liu_adatas.append(adata)
    print(f"  -> {adata.n_obs:,} cells, {adata.n_vars:,} genes")

# Concatenate
adata_liu = ad.concat(liu_adatas, join='outer')
adata_liu.obs_names_make_unique()
adata_liu.var_names_make_unique()
print(f"\nLiu combined: {adata_liu.n_obs:,} cells, {adata_liu.n_vars:,} genes")
print(adata_liu.obs['Type'].value_counts())

Loading KoSkin1...
filtered out 12341 genes that are detected in less than 3 cells
  QC: 8,761 -> 8,378 cells, 32,228 -> 19,887 genes
  -> 8,378 cells, 19,887 genes
Loading KoWound1...
filtered out 12155 genes that are detected in less than 3 cells
  QC: 8,743 -> 8,318 cells, 32,228 -> 20,073 genes
  -> 8,318 cells, 20,073 genes
Loading WtSkin1...
filtered out 11879 genes that are detected in less than 3 cells
  QC: 10,185 -> 9,718 cells, 32,228 -> 20,349 genes
  -> 9,718 cells, 20,349 genes
Loading WtWound1...
filtered out 11668 genes that are detected in less than 3 cells
  QC: 8,720 -> 8,135 cells, 32,228 -> 20,560 genes
  -> 8,135 cells, 20,560 genes

Liu combined: 34,549 cells, 21,811 genes
Type
WT_Skin     9718
KO_Skin     8378
KO_Wound    8318
WT_Wound    8135
Name: count, dtype: int64


In [23]:
# Export Liu
export_to_internal_format(adata_liu, "liu_snhg26", DATA_DIR)

  Wrote counts_liu_snhg26.mtx (21811 genes x 34549 cells)
  Wrote gene_names_liu_snhg26.csv (21811 genes)
  Wrote metadata_liu_snhg26.csv (34549 cells, columns: ['Sample', 'Genotype', 'Tissue', 'Type'])


(PosixPath('data/counts_liu_snhg26.mtx'),
 PosixPath('data/gene_names_liu_snhg26.csv'),
 PosixPath('data/metadata_liu_snhg26.csv'))

---
## 5. GSE178758 - Foster (Dermal Fibroblast scRNA-seq)

Integrated spatial multi-omics reveals fibroblast fate during wound healing.  
scRNA-seq only: 11 samples (Inner/Outer wound, POD2/7/14).  
Data format: processed CSV expression matrix (`GSE178758_scRNA-seq_processed.csv.gz`).  

Cell IDs encode sample info: `<well>.<celltype>.<sampleID>.<barcode>...`

In [24]:
foster_dir = DATA_DIR / "DeshkaS-Foster-GSE178758"
foster_csv = foster_dir / "GSE178758_scRNA-seq_processed.csv.gz"
print(f"Foster CSV: {foster_csv} (exists: {foster_csv.exists()})")

Foster CSV: data/DeshkaS-Foster-GSE178758/GSE178758_scRNA-seq_processed.csv.gz (exists: True)


In [25]:
# Sample ID -> metadata mapping (from GSE178758 series matrix)
foster_sample_metadata = {
    '18101611': {'region': 'Outer', 'timepoint': 'POD7'},
    '18101612': {'region': 'Outer', 'timepoint': 'POD7'},
    '18101614': {'region': 'Outer', 'timepoint': 'POD2'},
    '18101615': {'region': 'Outer', 'timepoint': 'POD2'},
    '19031328': {'region': 'Outer', 'timepoint': 'POD2'},
    '19031326': {'region': 'Outer', 'timepoint': 'POD14'},
    '18101613': {'region': 'Inner', 'timepoint': 'POD7'},
    '18101621': {'region': 'Inner', 'timepoint': 'POD2'},
    '18101622': {'region': 'Inner', 'timepoint': 'POD2'},
    '18092509': {'region': 'Inner', 'timepoint': 'POD14'},
    '18092510': {'region': 'Inner', 'timepoint': 'POD14'},
}

# Load CSV (genes are rows, cells are columns -> transpose)
print("Loading Foster CSV...")
df = pd.read_csv(foster_csv, compression='gzip', index_col=0)
print(f"Raw CSV shape (genes x cells): {df.shape}")
df = df.T  # Now cells x genes
print(f"Transposed (cells x genes): {df.shape}")

Loading Foster CSV...
Raw CSV shape (genes x cells): (15360, 191)
Transposed (cells x genes): (191, 15360)


In [26]:
# Parse cell IDs to extract metadata and filter to scRNA-seq samples
metadata_records = []
valid_cells = []

for cell_id in df.index:
    parts = cell_id.split('.')
    if len(parts) >= 3:
        well = parts[0]
        cell_type = parts[1]
        sample_id = parts[2]

        if sample_id in foster_sample_metadata:
            meta = foster_sample_metadata[sample_id]
            metadata_records.append({
                'cell_id': cell_id,
                'well': well,
                'cell_type_orig': cell_type,
                'sample_id': sample_id,
                'Sample': f"{meta['region']}_{meta['timepoint']}_{sample_id}",
                'region': meta['region'],
                'Timepoint': meta['timepoint'],
                'Type': f"{meta['region']}_{meta['timepoint']}",
            })
            valid_cells.append(cell_id)

print(f"Valid scRNA-seq cells: {len(valid_cells)} / {len(df)}")

# Filter and build AnnData
df_filtered = df.loc[valid_cells]
metadata_df = pd.DataFrame(metadata_records).set_index('cell_id')
metadata_df = metadata_df.loc[df_filtered.index]

adata_foster = ad.AnnData(
    X=csr_matrix(df_filtered.values.astype(np.float32)),
    obs=metadata_df,
    var=pd.DataFrame(index=df_filtered.columns),
)
adata_foster.obs_names = [f"Foster_{i}" for i in range(adata_foster.n_obs)]
adata_foster.var_names_make_unique()

print(f"\nBefore QC: {adata_foster.n_obs:,} cells, {adata_foster.n_vars:,} genes")
adata_foster = basic_qc_filter(adata_foster)
print(f"After QC: {adata_foster.n_obs:,} cells, {adata_foster.n_vars:,} genes")
print(adata_foster.obs['Type'].value_counts())

Valid scRNA-seq cells: 105 / 191

Before QC: 105 cells, 15,360 genes
filtered out 3645 genes that are detected in less than 3 cells
  QC: 105 -> 105 cells, 15,360 -> 11,715 genes
After QC: 105 cells, 11,715 genes
Type
Outer_POD2     43
Inner_POD14    23
Outer_POD7     22
Inner_POD2     14
Inner_POD7      3
Name: count, dtype: int64


In [27]:
# Export Foster
export_to_internal_format(adata_foster, "foster_scrna", DATA_DIR)

  Wrote counts_foster_scrna.mtx (11715 genes x 105 cells)
  Wrote gene_names_foster_scrna.csv (11715 genes)
  Wrote metadata_foster_scrna.csv (105 cells, columns: ['well', 'cell_type_orig', 'sample_id', 'Sample', 'region', 'Timepoint', 'Type'])


(PosixPath('data/counts_foster_scrna.mtx'),
 PosixPath('data/gene_names_foster_scrna.csv'),
 PosixPath('data/metadata_foster_scrna.csv'))

---
## Summary

All GEO datasets have been exported in the internal format.

In [28]:
# Verify all exported files exist
geo_datasets = [
    'haensel_wound_d4',
    'guerrero_wound_d12',
    'mascharak_regeneration',
    'liu_snhg26',
    'foster_scrna',
]

print("=" * 70)
print("Exported GEO Dataset Files")
print("=" * 70)

for name in geo_datasets:
    counts_f = DATA_DIR / f"counts_{name}.mtx"
    genes_f = DATA_DIR / f"gene_names_{name}.csv"
    meta_f = DATA_DIR / f"metadata_{name}.csv"
    
    print(f"\n{name}:")
    for label, path in [('counts', counts_f), ('genes', genes_f), ('metadata', meta_f)]:
        if path.exists():
            size_mb = path.stat().st_size / 1e6
            print(f"  {label}: {path.name} ({size_mb:.1f} MB)")
        else:
            print(f"  {label}: MISSING - {path.name}")

print("\n" + "=" * 70)
print("Done! These files can now be loaded in 01_scvi_integration.ipynb")
print("using the same DATASET_CONFIG and load_dataset_from_mtx() as internal datasets.")
print("=" * 70)

Exported GEO Dataset Files

haensel_wound_d4:
  counts: counts_haensel_wound_d4.mtx (723.0 MB)
  genes: gene_names_haensel_wound_d4.csv (0.1 MB)
  metadata: metadata_haensel_wound_d4.csv (1.4 MB)

guerrero_wound_d12:
  counts: counts_guerrero_wound_d12.mtx (333.2 MB)
  genes: gene_names_guerrero_wound_d12.csv (0.1 MB)
  metadata: metadata_guerrero_wound_d12.csv (1.2 MB)

mascharak_regeneration:
  counts: counts_mascharak_regeneration.mtx (224.9 MB)
  genes: gene_names_mascharak_regeneration.csv (0.1 MB)
  metadata: metadata_mascharak_regeneration.csv (0.5 MB)

liu_snhg26:
  counts: counts_liu_snhg26.mtx (1266.7 MB)
  genes: gene_names_liu_snhg26.csv (0.2 MB)
  metadata: metadata_liu_snhg26.csv (1.8 MB)

foster_scrna:
  counts: counts_foster_scrna.mtx (2.5 MB)
  genes: gene_names_foster_scrna.csv (0.1 MB)
  metadata: metadata_foster_scrna.csv (0.0 MB)

Done! These files can now be loaded in 01_scvi_integration.ipynb
using the same DATASET_CONFIG and load_dataset_from_mtx() as internal d